 # Documentación Técnica del Código: Arquitectura y Decisiones de Diseño
 Este cuaderno explica paso a paso **qué hace cada módulo** del proyecto y **por qué se ha programado de esa manera**, facilitando el mantenimiento y la escalabilidad por parte de otros desarrolladores.

## 1. Importación y Configuración de Rutas (`variables.py`)
**¿Por qué usamos `pathlib` y constantes tipadas?**
En proyectos colaborativos, las rutas escritas como texto (`"src/data/archivo.csv"`) fallan al pasar de macOS/Linux a Windows debido al separador de directorios. `Path(__file__).resolve().parent` detecta dinámicamente la ubicación real del script en el disco duro.

In [ ]:
import sys
from pathlib import Path

# Añadimos el directorio 'src' al path para importar nuestros módulos locales limpiamente
ruta_src = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ruta_src) not in sys.path:
    sys.path.append(str(ruta_src))

import variables as var
print(f"Ruta de datos configurada en: {var.RUTA_DATOS}")
print(f"Activos analizados: {list(var.TICKERS_ACTIVOS.keys())}")

Ruta de datos configurada en: C:\Users\i-ote\Documents\GitHub\Proyecto_EDA\src\data
Activos analizados: ['SPY', 'QQQ', '^IBEX', 'GC=F', 'SHY', 'BTC-USD']


## 2. Ingesta Resiliente y Limpieza sin Sesgo de Futuro (`utils.py` y `funciones.py`)
**¿Por qué no usamos la media (`.mean()`) para rellenar nulos en finanzas?**
Si rellenamos un dato faltante de marzo de 2024 con la media de todo el periodo 2024-2026, estamos introduciendo información del año 2025 en el año 2024 (*look-ahead bias*). En los mercados reales, si un mercado cierra por festivo, el valor vigente del activo sigue siendo el del último cierre disponible. Por eso aplicamos `.ffill()` (*Forward Fill*).

In [ ]:
from utils.utils import inicializar_directorios, obtener_datos_financieros
from funciones import limpiar_y_preprocesar_datos

inicializar_directorios([var.RUTA_DATOS, var.RUTA_GRAFICOS])
df_bruto = obtener_datos_financieros(var.TICKERS_ACTIVOS, var.FECHA_INICIO, var.FECHA_FIN, var.ARCHIVO_DATOS_BRUTOS)
df_limpio = limpiar_y_preprocesar_datos(df_bruto, var.ARCHIVO_DATOS_LIMPIOS)
df_limpio.head()

[INFO] Cargando datos brutos desde almacenamiento local: activos_brutos.csv
[LIMPIEZA] Estado inicial -> Nulos: 20 | Fechas duplicadas: 2
[LIMPIEZA] Estado final   -> Nulos: 0 | Filas válidas: 524


,SPY,QQQ,^IBEX,GC=F,SHY,BTC-USD
Date,,,,,,
2024-01-01,471.05,413.06,10097.88,2060.29,82.15,42444.84
2024-01-02,465.76,420.45,10010.75,2042.28,82.16,49251.41
2024-01-03,467.02,425.18,10127.37,2062.02,82.12,48476.94
2024-01-04,467.03,425.20,10441.32,2056.20,82.19,48885.49
2024-01-05,472.68,424.59,10358.33,2032.51,82.32,48441.13


## 3. Vectorización de Rentabilidades y Estadísticos (`funciones.py`)
**¿Por qué usamos `.pct_change(periods=21)` en lugar de bucles `for`?**
Un bucle `for` en Python itera fila por fila interpretando tipos dinámicamente. El método `.pct_change()` delega el cálculo a arrays contiguos de memoria en C a través de NumPy y Pandas, logrando máxima eficiencia computacional.

In [ ]:
from funciones import calcular_retornos, obtener_estadisticos_descriptivos

ret_diarios, ret_corto_plazo = calcular_retornos(df_limpio, var.VENTANA_CORTO_PLAZO)
resumen = obtener_estadisticos_descriptivos(
    ret_diarios, ret_corto_plazo, var.TASA_LIBRE_RIESGO_ANUAL,
    var.DIAS_BURSATILES_ANIO, var.NIVEL_CONFIANZA_VAR, var.ARCHIVO_ESTADISTICOS
)
resumen

,Retorno_Medio_Mensual_%,Retorno_Mediano_Mensual_%,Mejor_Mes_%,Peor_Mes_%,Probabilidad_Ganancia_Mensual_%,VaR_95_Mensual_%,Retorno_Anualizado_%,Volatilidad_Anualizada_%,Ratio_Sharpe,Max_Drawdown_%,Asimetria_Skewness,Curtosis
GC=F,1.60,1.91,11.13,-8.33,68.99,-4.73,21.49,12.07,1.51,-9.69,-0.01,0.53
SPY,1.54,1.72,10.28,-7.58,65.81,-4.56,16.68,13.15,1.02,-8.13,-0.07,-0.06
SHY,0.36,0.37,1.30,-0.64,75.35,-0.32,4.53,1.51,0.85,-1.05,0.05,-0.20
^IBEX,1.17,0.95,11.83,-7.56,58.05,-4.89,14.95,14.44,0.81,-10.92,0.13,0.43
QQQ,2.04,2.24,22.94,-17.78,63.62,-8.69,20.29,22.40,0.76,-27.51,0.03,0.21
BTC-USD,3.35,-1.59,68.44,-38.53,47.32,-27.11,58.22,84.62,0.65,-60.75,6.45,96.55


## 4. Conexión a la API: El papel de yfinance (`utils.py`)
La conexión a la API externa no aparece escrita como un `requests.get(url)` explícito porque está encapsulada dentro de la función `yf.download(...)` en el módulo src/utils/utils.py, mientras que la creación física del archivo `activos_brutos.csv` ocurre justo al final de esa misma función mediante `df_bruto.to_csv(ruta_guardado)`.

`yfinance` actúa como una librería envoltorio (wrapper) de alto nivel. Si tuviéramos que conectarnos manualmente a los servidores de Yahoo Finance usando la librería estándar `requests`, tendríamos que programar las peticiones HTTP, gestionar cabeceras (*User-Agent*), negociar cookies de sesión y transformar un archivo JSON complejo en una tabla.

La función `yf.download(...)` realiza todo ese trabajo por debajo de forma automática:
1. Construye internamente la URL de la API REST de Yahoo Finance para cada activo  
Por ejemplo: [https://query2.finance.yahoo.com/v8/finance/chart/SPY](https://query2.finance.yahoo.com/v8/finance/chart/SPY).
2. Ejecuta las peticiones HTTP GET mediante requests / curl_cffi.
3. Descarga la respuesta en formato JSON, extrae los precios de cierre ajustados (Adjusted Close) y los devuelve directamente estructurados como un DataFrame de Pandas.

Si se quisiera realizar la consulta sin utilizar yfinance y se quisira emplear directamente `requests`, el código equivalente para un solo activo sería el siguiente:
```python
import requests

url = "https://query2.finance.yahoo.com/v8/finance/chart/SPY?interval=1d&range=1mo"
cabeceras = {"User-Agent": "Mozilla/5.0"}

respuesta = requests.get(url, headers=cabeceras)
datos_json = respuesta.json()  # Devuelve un diccionario anidado que habría que parsear manualmente
```

Como se puede apreciar, utilizar `yfinance` nos ahorra tener que recorrer manualmente las claves internas del JSON (chart -> result -> indicators -> adjclose) y nos permite descargar los seis activos simultáneamente en una sola instrucción limpia y eficiente.

#### Sistema de caché local ( `if ruta_guardado.exists():` )

Al inicio de `obtener_datos_financieros(...)` incluimos esta comprobación:

* **¿Qué implica esto?**: Si ya se guardó en la carpeta src/data/ el `archivo activos_brutos.csv`, cuando se ejecute `python src/main.py` el programa detectará que el archivo ya existe en el disco duro y lo leerá directamente sin salir a Internet (patrón Cache-Aside).
* **¿Cómo forzar la descarga real desde Internet?** Si se desea ver en vivo cómo el programa se conecta a la API de Yahoo Finance y crea un activos_brutos.csv nuevo con los datos reales de mercado, simplemente elimine (o mueva) el archivo src/data/activos_brutos.csv y vuelva a ejecutar python src/main.py en la terminal de Visual Studio Code con conexión a Internet. Verá en consola el mensaje [INFO] Conectando con la API de Yahoo Finance....